# Granite Peak CRE: 07 Genie space and benchmarks

Creates (or updates) the **Granite Peak Portfolio Intelligence** Genie space from version-controlled files:

| File | Content |
|---|---|
| `src/genie/instructions.md` | Text instructions: reporting date, measures, clause coverage rule, H3 distance rules, names |
| `src/genie/examples.json` | Example question → SQL pairs that teach query patterns |
| `src/genie/benchmarks.json` | 11 benchmark questions with validated SQL (also loaded as space benchmarks) |
| `src/genie/functions.sql` | Trusted H3 functions (created by notebook 04) |

The notebook then asks Genie every benchmark question through the Conversation API, runs the validated SQL, compares the two results and writes a scored report to `artifacts/evidence/genie_benchmarks.json`.

In [1]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
dbutils.widgets.text("warehouse_id", "", "SQL warehouse (blank = first serverless)")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
uc = f"{catalog}.{schema}"
TITLE = "Granite Peak Portfolio Intelligence"

import json, os, re, secrets
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src", "genie"))
def src(name):
    return open(os.path.join(SRC, name)).read().replace("${uc_schema}", uc)

warehouse_id = dbutils.widgets.get("warehouse_id").strip() or next(
    wh.id for wh in w.warehouses.list() if wh.enable_serverless_compute)
print(f"Target: {uc}  |  warehouse: {warehouse_id}")

Target: smmorgan_catalog.granite_commercial_real_estate  |  warehouse: ebe7d321697a8988


In [2]:
COLUMN_NOTES = {  # column descriptions that disambiguate similar measures
    "h3_market_context": {
        "retail_count": "Retail store POIs within ~1 km; use for questions about retail POIs or retail competition",
        "food_and_beverage_count": "Restaurant, cafe and bar POIs within ~1 km",
        "office_count": "Office and professional-service POIs within ~1 km",
        "industrial_count": "Manufacturer, distributor and logistics POIs within ~1 km",
        "amenity_count": "Parks, gyms, hotels, schools and clinics within ~1 km",
        "competitor_count": "POIs that compete with the property's own type (retail + food for Retail, office for Office, industrial for Industrial); use only when no POI category is named",
    },
    "rent_roll": {
        "termination_right_date": "Earliest date the tenant may terminate; 'termination right in the next N months' filters on this date",
        "months_to_expiry": "Months from the reporting date (2026-10-01) to lease_end",
    },
}

TABLES = {
    "rent_roll":             "Active leases: ERP terms plus document-derived clauses; row-filtered by fund",
    "lease_abstract":        "Fields extracted from lease PDFs (ai_parse_document + ai_extract) with analyst corrections",
    "lease_discrepancies":   "Leases whose PDF terms disagree with the ERP",
    "renewal_scores":        "Renewal probability and expected vacancy loss per active lease (ML model)",
    "deal_pipeline":         "Acquisition and disposition pipeline with stage, ask price, cap rate, IRR and H3 cells",
    "deal_lease_terms":      "In-place leases extracted from deal offering memoranda and rent rolls",
    "property_360":          "One row per property: occupancy, ABR, WALT, rollover, competitors, neighborhood",
    "property_neighborhood": "Neighborhood, macrohood and locality containing each property (exact polygon test)",
    "h3_market_context":     "Overture POI counts within ~1 km (H3 k-ring 3) of each property",
    "rollover_schedule":     "ABR and square feet expiring per quarter by fund, property type and MSA",
}
METRIC_VIEWS = {
    "portfolio_metrics": "Lease KPIs: ABR, WALT, lease count, rollover within 24 months, abstraction coverage",
    "occupancy_metrics": "Monthly occupancy rate by region, MSA, property type and fund",
}
FUNCTIONS = {
    "assets_near_deal":   "Held assets near a pipeline deal with occupancy and ABR: assets_near_deal(deal_id, k_ring)",
    "assets_near":        "Assets near a longitude/latitude point: assets_near(lon, lat, k_ring)",
    "competitors_within": "POI counts by group around a property: competitors_within(property_id, k_ring)",
}

JOINS = [  # (left, right, key, relationship)
    ("rent_roll", "h3_market_context", "property_id", "MANY_TO_ONE"),
    ("rent_roll", "renewal_scores", "lease_id", "ONE_TO_ONE"),
    ("rent_roll", "lease_abstract", "lease_id", "ONE_TO_ONE"),
    ("rent_roll", "property_360", "property_id", "MANY_TO_ONE"),
    ("property_360", "property_neighborhood", "property_id", "ONE_TO_ONE"),
    ("property_360", "h3_market_context", "property_id", "ONE_TO_ONE"),
]

def sql_lines(sql):
    lines = sql.strip().split("\n")
    return [l + "\n" for l in lines[:-1]] + [lines[-1]]

def hex_id():
    return secrets.token_hex(16)

examples = json.loads(src("examples.json"))
benchmarks = json.loads(src("benchmarks.json"))
instructions = src("instructions.md").strip().split("\n")

space = {
    "version": 2,
    "config": {"sample_questions": sorted([{"id": hex_id(), "question": [b["question"]]} for b in benchmarks[:5]],
                                          key=lambda x: x["id"])},
    "data_sources": {
        "tables": sorted([{"identifier": f"{uc}.{t}", "description": [d],
                           **({"column_configs": sorted([{"column_name": col, "description": [note]}
                                                         for col, note in COLUMN_NOTES[t].items()],
                                                        key=lambda x: x["column_name"])} if t in COLUMN_NOTES else {})}
                          for t, d in TABLES.items()],
                         key=lambda x: x["identifier"]),
        "metric_views": sorted([{"identifier": f"{uc}.{m}", "description": [d]} for m, d in METRIC_VIEWS.items()],
                               key=lambda x: x["identifier"]),
    },
    "instructions": {
        "text_instructions": [{"id": hex_id(), "content": [l + "\n" for l in instructions]}],
        "example_question_sqls": sorted([{"id": hex_id(), "question": [e["question"]], "sql": sql_lines(e["sql"]),
                                          "usage_guidance": [e["usage_guidance"]]} for e in examples],
                                        key=lambda x: x["id"]),
        "sql_functions": sorted([{"id": hex_id(), "identifier": f"{uc}.{f}"} for f in FUNCTIONS],
                                key=lambda x: (x["id"], x["identifier"])),
        "join_specs": sorted([{"id": hex_id(),
                               "left": {"identifier": f"{uc}.{l}", "alias": l},
                               "right": {"identifier": f"{uc}.{r}", "alias": r},
                               "sql": [f"`{l}`.`{k}` = `{r}`.`{k}`", f"--rt=FROM_RELATIONSHIP_TYPE_{rel}--"],
                               "comment": [f"Join {l} to {r} on {k}"]} for l, r, k, rel in JOINS],
                             key=lambda x: x["id"]),
    },
    "benchmarks": {"questions": sorted([{"id": hex_id(), "question": [b["question"]],
                                         "answer": [{"format": "SQL", "content": sql_lines(b["sql"])}]}
                                        for b in benchmarks], key=lambda x: x["id"])},
}
print(f"tables {len(TABLES)}  metric views {len(METRIC_VIEWS)}  functions {len(FUNCTIONS)}  "
      f"joins {len(JOINS)}  examples {len(examples)}  benchmarks {len(benchmarks)}  instruction lines {len(instructions)}")

tables 10  metric views 2  functions 3  joins 6  examples 4  benchmarks 11  instruction lines 36


In [3]:
existing = [s for s in w.api_client.do("GET", "/api/2.0/genie/spaces").get("spaces", []) if s.get("title") == TITLE]
body = {"title": TITLE, "warehouse_id": warehouse_id, "serialized_space": json.dumps(space),
        "description": "Granite Peak CRE portfolio, leases, deals and H3 market context (synthetic + Overture open data)"}
if existing:
    space_id = existing[0]["space_id"]
    w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{space_id}", body=body)
    print(f"updated Genie space {space_id}")
else:
    me = w.current_user.me().user_name
    space_id = w.api_client.do("POST", "/api/2.0/genie/spaces", body={**body, "parent_path": f"/Users/{me}"})["space_id"]
    print(f"created Genie space {space_id}")

groups = ["granite_portfolio_leadership", "granite_fund_i_analysts", "granite_fund_ii_analysts",
          "granite_fund_iii_analysts", "granite_fund_iv_analysts"]
w.api_client.do("PATCH", f"/api/2.0/permissions/genie/{space_id}",
                body={"access_control_list": [{"group_name": g, "permission_level": "CAN_RUN"} for g in groups]})
print("granted CAN_RUN on the space to", ", ".join(groups))
# Unity Catalog grants need account-level groups; these demo groups are workspace-local, so table access
# for fund analysts would be granted to the equivalent account groups. The row filter (is_member) works with both.
dbutils.jobs.taskValues.set("genie_space_id", space_id)

updated Genie space 01f1ce5643ff1c4bafd90cea16c1fba7
granted CAN_RUN on the space to granite_portfolio_leadership, granite_fund_i_analysts, granite_fund_ii_analysts, granite_fund_iii_analysts, granite_fund_iv_analysts


In [4]:
import time
from decimal import Decimal

def norm(v):
    if v is None:
        return "null"
    if isinstance(v, bool) or str(v).lower() in ("true", "false"):
        return str(v).lower()
    try:
        f = float(v)
        return f"{f:.2f}"
    except (TypeError, ValueError):
        return str(v).strip().lower()

def rows_match(expected, actual):
    """Each expected row's values must appear within some actual row (extra columns allowed)."""
    pool = [set(map(norm, r)) for r in actual]
    hits = 0
    for r in expected:
        want = set(map(norm, r))
        # Accept fractions reported as percentages (0.23 vs 23.00)
        pct = {f"{float(v) * 100:.2f}" if v.replace(".", "", 1).replace("-", "", 1).isdigit() and abs(float(v)) <= 1 else v
               for v in want}
        for i, have in enumerate(pool):
            if want <= have or pct <= have or {v for v in want if v != "null"} <= have:
                hits += 1; pool.pop(i); break
    return hits

def ask(question):
    msg = w.genie.start_conversation_and_wait(space_id, question)
    sql, rows, text = None, [], None
    for att in msg.attachments or []:
        if att.text:
            text = att.text.content
        if att.query:
            sql = att.query.query
            res = w.genie.get_message_attachment_query_result(space_id, msg.conversation_id, msg.message_id, att.attachment_id)
            rows = res.statement_response.result.data_array or []
    return sql, rows, text

results = []
for b in benchmarks:
    expected = [list(r) for r in spark.sql(b["sql"]).collect()]
    try:
        sql, actual, text = ask(b["question"])
        error = None
    except Exception as e:  # keep scoring the rest
        sql, actual, text, error = None, [], None, str(e)[:300]
    hits = rows_match(expected, actual)
    verdict = ("PASS" if expected and hits == len(expected) and len(actual) == len(expected)
               else "PARTIAL" if hits > 0 else "FAIL")
    results.append({"id": b["id"], "question": b["question"], "verdict": verdict,
                    "expected_rows": len(expected), "genie_rows": len(actual), "matched_rows": hits,
                    "genie_sql": sql, "expected_sql": b["sql"], "genie_text": text, "error": error,
                    "expected_sample": [[norm(v) for v in r] for r in expected[:5]],
                    "genie_sample": [[norm(v) for v in r] for r in actual[:5]]})
    print(f"{b['id']} {verdict:7s} expected {len(expected):3d}  genie {len(actual):3d}  matched {hits:3d}  {b['question']}")
    time.sleep(2)

passed = sum(r["verdict"] == "PASS" for r in results)
print(f"\nBenchmark accuracy: {passed}/{len(results)} PASS ({passed / len(results):.0%}), "
      f"{sum(r['verdict'] == 'PARTIAL' for r in results)} PARTIAL")

b01 PASS    expected   7  genie   7  matched   7  Show acquisitions in the pipeline over $50M by property type and market.
b02 PARTIAL expected  57  genie  57  matched  10  Which held assets have more than 20% of rent rolling in the next 24 months?
b03 PASS    expected   6  genie   6  matched   6  Compare occupancy this year vs last year by region.
b04 PASS    expected   5  genie   5  matched   5  Which retail assets have more than 25% of ABR expiring within 24 months and a co-tenancy clause?
b05 PASS    expected  40  genie  40  matched  40  For every property with a lease expiring in 2027, how many retail POIs are within about 1 km?
b06 PASS    expected   8  genie   8  matched   8  Show leases where the abstracted expiration date disagrees with the ERP.
b07 PASS    expected  31  genie  31  matched  31  What's the WALT by fund and MSA?
b08 PASS    expected  10  genie  10  matched  10  Which tenants with termination rights in the next 18 months have renewal probability below 40%?
b09 PA

In [5]:
report = {"space_title": TITLE, "benchmarks": len(results), "pass": passed,
          "partial": sum(r["verdict"] == "PARTIAL" for r in results),
          "accuracy": round(passed / len(results), 3), "results": results}
out_dir = f"/Volumes/{catalog}/{schema}/artifacts/evidence"
os.makedirs(out_dir, exist_ok=True)
with open(f"{out_dir}/genie_benchmarks.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
for r in results:
    print(f"--- {r['id']} [{r['verdict']}] {r['question']}\n{r['genie_sql']}\n")

--- b01 [PASS] Show acquisitions in the pipeline over $50M by property type and market.
SELECT
  `property_type`,
  `market`,
  COUNT(*) AS `deal_count`,
  SUM(`ask_price`) AS `total_ask_price`,
  AVG(`ask_price`) AS `avg_ask_price`
FROM `smmorgan_catalog`.`granite_commercial_real_estate`.`deal_pipeline`
WHERE `deal_type` ILIKE '%acq%'
  AND (`stage` ILIKE '%screening%'
    OR `stage` ILIKE '%loi%'
    OR `stage` ILIKE '%due diligence%'
    OR `stage` ILIKE '%under contract%')
  AND `ask_price` > 50000000
  AND `property_type` IS NOT NULL
  AND `market` IS NOT NULL
GROUP BY `property_type`, `market`
ORDER BY `total_ask_price` DESC, `deal_count` DESC

--- b02 [PARTIAL] Which held assets have more than 20% of rent rolling in the next 24 months?
SELECT
  `property_id`,
  `property_name`,
  `property_type`,
  `msa`,
  `state`,
  MEASURE(`abr`) AS `abr`,
  MEASURE(`abr_expiring_24m`) AS `abr_expiring_24m`,
  MEASURE(`pct_abr_expiring_24m`) * 100 AS `pct_abr_expiring_24m`
FROM `smmorgan_cata